# **Routing Pipeline**

Solves the FL-MILP for one batch: it selects the models to provision, $M^* \subseteq M$ ($y_j$), and assigns each request to one of them, $r_i \rightarrow m_j$ ($x_{ij}$), so that the total energy is minimal under the quality, budget and capacity constraints. Decisions use the estimated output lengths; the chosen solutions are also recomputed with the reference lengths of the Magpie requests. The same batch is solved with the six baselines and the relaxed lower bound. The experiments notebook runs this notebook with the parameters of each run.

## Setup

In [1]:
!pip install -q "pulp==2.9.0" pyarrow

In [ ]:
import json
import os
import time

import numpy as np
import pandas as pd
import pulp

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 250)
pd.set_option("display.max_colwidth", 60)

def show(df, title=None):
    # tables are printed as plain text so that they stay in the saved notebook
    if title:
        print(f"\n=== {title}")
    print(df.to_string() if hasattr(df, "to_string") else df)

def short(model_key):
    # "meta-llama/llama-3.1-8b-instruct" -> "llama-3.1-8b-instruct"
    return model_key.split("/")[1]

## Configuration

In [ ]:
SEED = 42

# input and output files
POOL_VERSION = "v1.3"
MODEL_POOL_PATH = f"data/model_pool/model_pool_core_{POOL_VERSION}.parquet"
BATCH_CURVE_PATH = f"data/model_pool/mlenergy_batch_curve_{POOL_VERSION}.parquet"
REQUEST_POOL_PATH = "data/request_pool/request_pool_v2.parquet"
ESTIMATES_PATH = "data/request_estimates/request_estimates_v1.parquet"
OUTPUT_DIR = "results/pipeline"

# batch
N_REQUESTS = 200
DOMAIN_MIX = None             # e.g. {"natural_sciences": 0.5, "general": 0.5}; None = as in the pool
TIER_MIX = {"Economy": 0.2, "Standard": 0.6, "Premium": 0.2}   # share of requests per service tier (drawn at random)
EXCLUDE_SOURCES = ["longbench_v1", "longbench_v2"]   # long-context requests excluded (length estimate and energy model unreliable on very long inputs)
REFERENCE_SOURCE = "magpie"        # only source whose reference answers are generated by an LLM: the only reference lengths used for evaluation
FAILED_GENERATION_TOKENS = 3900   # Magpie references this long are degenerate (loops): treated as missing (empirical threshold)

# quality
QUALITY_SCALE = "absolute"    # "absolute": Open LLM scores/100; "relative": divided by the best model of the pool
W_DOMAIN, W_TASK = 0.5, 0.5   # weights of the domain score and of the task score
ALPHA_COMPLEXITY = 0.5        # weight of the hard score for the hardest requests
DEFAULT_COMPLEXITY = 0.5      # requests whose source has no difficulty label
TIER_RATIO = {"Economy": 0.50, "Standard": 0.75, "Premium": 0.95}   # q_min = ratio × best quality of the request

# time and energy
WORKLOAD_DURATION_S = 600     # T: period in which the batch is served (10 minutes)
BATCH = 128                   # batch capacity: requests processed in parallel on the GPU; ML.ENERGY coefficients at this batch size (offline, saturated batches)
ENERGY_ACCOUNTING = "marginal"    # fixed choice: energy above a GPU that is on but has no model loaded
ALREADY_ACTIVE = []           # models already loaded when the batch arrives (a_j = 1)
ENFORCE_CAPACITY = True       # serving + loading must fit in T
SOLVER_TIME_LIMIT_S = 600     # time limit of the main MILP ("Solve")
DIAGNOSIS_TIME_LIMIT_S = 60   # time limit of each attempt of the infeasibility diagnosis ("Solve")
INPUT_OUTPUT_RATIO = 150      # an input token costs 1/150 of an output token (LLM-Perf median)
USE_VERBOSITY = True          # True: L_ij = L_i × v_j (main case); False: every model writes the estimated length (v_j = 1, sensitivity)
VERBOSITY_BATCH = 128         # v_j is read from the ML.ENERGY row at this batch capacity, whatever BATCH is (answer lengths do not depend on it)
VERBOSITY_REFERENCE = "meta-llama/llama-3.1-70b-instruct"   # v_j = ML.ENERGY length of j / length of this model (Magpie answers, used to train the length estimator, were written by Llama-3.1-70B)

# Vadari (2026), H100 SXM, per GPU
H100_NO_MODEL_W = 71.8        # P0: GPU on, no model loaded
H100_IDLE_WITH_MODEL_W = 121.7    # GPU on, model loaded, no requests (parking)
H100_LOADING_W = 124.0        # while the model is being loaded
# Loading time = fixed part + weights/speed: line through two points of Vadari
# 15 GB in 30 s (Qwen2.5-7B, measured) and 141 GB in 45 s (70B scenario)
LOAD_FIXED_S = 28.2
LOAD_GB_PER_S = 8.4

# cost
COST_MODE = "market"          # fixed choice: price per token; "gpu_rental" kept only as an unused alternative
H100_USD_PER_HOUR = 2.74      # used only by "gpu_rental"
BUDGET_MARGIN = 0.20          # delta: B = (1 + delta) × C_min_ref

# reference configuration of the budget
# budget is a property of the batch: C_min_ref is computed on the requests of the batch with these values
# a run that changes TIER_RATIO, TIER_MIX, W_DOMAIN, W_TASK or ALPHA_COMPLEXITY keeps the budget of its batch: one parameter changes at a time
REFERENCE_TIER_RATIO = {"Economy": 0.50, "Standard": 0.75, "Premium": 0.95}
REFERENCE_TIER_MIX = {"Economy": 0.2, "Standard": 0.6, "Premium": 0.2}
REFERENCE_W_DOMAIN, REFERENCE_W_TASK = 0.5, 0.5
REFERENCE_ALPHA_COMPLEXITY = 0.5

# specialization
SUBJECT_FOCUS = None          # e.g. "health": SUBJECT_FOCUS_SHARE of the requests are drawn from the requests of this MMLU-Pro subject only
SUBJECT_FOCUS_SHARE = 1.0     # the rest is drawn from the other requests, as in the pool; with replacement if the subject has too few requests
HYPOTHETICAL_SPECIALIST = None    # e.g. {"donor": "meta-llama/llama-3.1-8b-instruct", "subject": "health", "quality_ratio": 0.9}:
                              # a counterfactual model with the energy, throughput and price of the donor; on the requests of the subject its
                              # quality is quality_ratio x the best real model of the request, elsewhere the donor's quality. A scenario, not data
EXCLUDE_MODELS = []           # models left out of this run (e.g. a pool of one model generation); prices are still inherited from the whole pool

rng = np.random.default_rng(SEED)

## Load the data

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

models = pd.read_parquet(MODEL_POOL_PATH).rename(columns={"core_role": "role"}).set_index("model_key")
HS_KEY = None                 # key of the hypothetical specialist, if any
if HYPOTHETICAL_SPECIALIST:
    hs = HYPOTHETICAL_SPECIALIST
    assert hs["donor"] in models.index, f"donor {hs['donor']} not in the model pool"
    HS_KEY = f"hypothetical/{hs['subject']}-specialist"
    row = models.loc[hs["donor"]].copy()
    row["role"] = f"{hs['subject']} (hypothetical)"
    row["arch_fingerprint"] = "hypothetical: " + str(row["arch_fingerprint"])    # keeps the donor's price, never lends it to other models
    models.loc[HS_KEY] = row
    print(f"Hypothetical specialist {short(HS_KEY)}: energy, throughput and price of {short(hs['donor'])}; quality on "
          f"'{hs['subject']}' requests = {hs['quality_ratio']} x the best real model ('Expected quality'). A scenario, not data")
models_full = models.copy()   # the whole pool: prices of models without a provider are inherited from it
if EXCLUDE_MODELS:
    assert set(EXCLUDE_MODELS) <= set(models.index), f"unknown models in EXCLUDE_MODELS: {set(EXCLUDE_MODELS) - set(models.index)}"
    models = models.drop(index=list(EXCLUDE_MODELS))
    print(f"Models left out of this run: {[short(m) for m in EXCLUDE_MODELS]}; pool of the run: {[short(m) for m in models.index]}")
curve = pd.read_parquet(BATCH_CURVE_PATH)
request_pool = pd.read_parquet(REQUEST_POOL_PATH)
excluded = request_pool["source"].isin(EXCLUDE_SOURCES)
print(f"Excluded sources {EXCLUDE_SOURCES}: {int(excluded.sum())} requests")
request_pool = request_pool[~excluded].reset_index(drop=True)
estimates = pd.read_parquet(ESTIMATES_PATH)
request_pool = request_pool.merge(estimates[["request_id", "predicted_output_tokens"]], on="request_id", how="left")
print(f"Models: {len(models)} | batch curve: {curve['model_key'].nunique()} donors | "
      f"request pool: {len(request_pool)} | with estimates: {int(request_pool['predicted_output_tokens'].notna().sum())}")

Mounted at /content/drive
Excluded sources ['longbench_v1', 'longbench_v2']: 723 requests
Models: 9 | batch curve: 21 donors | request pool: 4319 | with estimates: 4319


## The two sets: models $M$ and requests $R$

### Models and their coefficients at the chosen batch

$$M = \{m_1, \dots, m_K\}$$

Energy per output token $\varepsilon_j$, throughput $\theta_j$, number of GPUs $g_j$ and mean answer length come from one row of the ML.ENERGY batch curve: the row of the energy donor at batch capacity `BATCH`.

In [ ]:
MODEL_IDS = list(models.index)
K = len(MODEL_IDS)
donor = models["mle_donor"].str.lower()
at_batch = curve[curve["batch"] == BATCH].set_index("model_key")
missing = sorted(set(donor) - set(at_batch.index))
assert not missing, f"no ML.ENERGY measurement at batch {BATCH} for {missing}; available: {sorted(curve['batch'].unique())}"

coef = at_batch.loc[donor.values, ["num_gpus", "j_per_out_token", "throughput_tok_s", "power_w", "avg_output_len"]]
coef.index = MODEL_IDS
# answer lengths read at VERBOSITY_BATCH for every batch capacity (the same prompts give the same answers)
at_verbosity_batch = curve[curve["batch"] == VERBOSITY_BATCH].set_index("model_key")
coef["avg_output_len"] = at_verbosity_batch.loc[donor.values, "avg_output_len"].values
REFERENCE_LENGTH = float(at_verbosity_batch.loc[VERBOSITY_REFERENCE, "avg_output_len"])   # from the curve: the reference model may be left out of the run
assert (coef["num_gpus"].values == models["mle_num_gpus"].values).all(), "GPUs of the curve differ from the pool"
show(pd.concat([models[["role", "params_b", "context_window", "mle_link"]], donor.map(short).rename("donor"),
                coef.round(3)], axis=1).rename(index=short), f"M: {K} models, coefficients at batch {BATCH}")


=== M: 9 models, coefficients at batch 128
                                           role   params_b  context_window   mle_link                   donor  num_gpus  j_per_out_token  throughput_tok_s   power_w  avg_output_len
llama3-openbiollm-70b                  medicine  70.000000          8192.0  inherited  llama-3.1-70b-instruct         4            0.563          3876.267  2183.755         358.930
llama3-med42-70b                       medicine  70.553706          8192.0  inherited  llama-3.1-70b-instruct         4            0.563          3876.267  2183.755         358.930
llama-3.1-70b-instruct               generalist  70.553706        131072.0     direct  llama-3.1-70b-instruct         4            0.563          3876.267  2183.755         358.930
llama-3.1-8b-instruct                generalist   8.030261        131072.0     direct   llama-3.1-8b-instruct         1            0.095          7036.532   669.244         404.590
llama-3.3-70b-instruct               generalist  70

### Requests

$$R = \{r_1, \dots, r_N\}$$

`predicted_output_tokens` is used for the decisions, `evaluation_output_tokens` only to recompute the solutions with the reference lengths of the Magpie requests. Each request receives a service tier drawn with the shares of `TIER_MIX`.

In [ ]:
import copy

request_pool["domain"] = request_pool["domain"].fillna("general")
assert request_pool["predicted_output_tokens"].notna().all(), "some requests have no output-length estimate"
failed = (request_pool["source"] == REFERENCE_SOURCE) & (request_pool["reference_output_tokens"] >= FAILED_GENERATION_TOKENS)
print(f"Magpie references treated as generation failures: {int(failed.sum())}")
request_pool["has_reference_length"] = (request_pool["source"] == REFERENCE_SOURCE) & request_pool["reference_output_tokens"].notna() & ~failed
request_pool["evaluation_output_tokens"] = request_pool["predicted_output_tokens"].astype(float).where(
    ~request_pool["has_reference_length"], request_pool["reference_output_tokens"].astype(float))

max_verbosity = (coef["avg_output_len"] / REFERENCE_LENGTH).max() if USE_VERBOSITY else 1.0
fits_some_model = (request_pool["input_tokens"] + request_pool["predicted_output_tokens"] * max_verbosity
                   <= models["context_window"].max())
candidates = request_pool[fits_some_model]
print(f"Requests too long for every model, never drawn: {int((~fits_some_model).sum())}")

# subjects of a request, defined here because SUBJECT_FOCUS draws the batch by subject
SUBJECTS_OF_DOMAIN = {
    "mathematics": ["math"],
    "coding": ["computer_science"],
    "natural_sciences": ["biology", "chemistry", "physics", "health"],
    "humanities": ["history", "philosophy"],
    "social_sciences": ["economics", "psychology", "law", "business"],
    "general": None,                       # None = all subjects
}
SUBJECT_OF_DEWEY = {"61": ["health"], "57": ["biology"], "59": ["biology"], "53": ["physics"], "54": ["chemistry"],
                    "62": ["engineering"], "51": ["math"], "00": ["computer_science"], "33": ["economics"],
                    "34": ["law"], "15": ["psychology"], "65": ["business"], "10": ["philosophy"],
                    "16": ["philosophy"], "17": ["philosophy"], "90": ["history"], "93": ["history"], "94": ["history"]}

def request_subjects(row):
    fine = str(row.get("domain_fine") or "")
    dewey = fine[:2] if fine[:2].isdigit() else None
    return SUBJECT_OF_DEWEY.get(dewey) or SUBJECTS_OF_DOMAIN.get(row["domain"])

def draw(pool_, n_):
    # without replacement; with replacement only if the pool has fewer requests than needed (stated)
    return pool_.sample(n_, replace=n_ > len(pool_), random_state=SEED)

if SUBJECT_FOCUS:
    in_focus = candidates.apply(request_subjects, axis=1).map(lambda s: s == [SUBJECT_FOCUS])
    focus_pool = candidates[in_focus]
    assert len(focus_pool), f"no request with subject {SUBJECT_FOCUS!r}"
    n_focus = int(round(SUBJECT_FOCUS_SHARE * N_REQUESTS))
    workload = pd.concat([draw(focus_pool, n_focus), draw(candidates[~in_focus], N_REQUESTS - n_focus)])
    print(f"SUBJECT_FOCUS = {SUBJECT_FOCUS!r}: {n_focus} of {N_REQUESTS} requests drawn from {len(focus_pool)} distinct "
          f"'{SUBJECT_FOCUS}' requests" + (" (with replacement)" if n_focus > len(focus_pool) else ""))
elif DOMAIN_MIX:
    parts = [candidates[candidates["domain"] == d].sample(
                 min(int(round(s * N_REQUESTS)), int((candidates["domain"] == d).sum())), random_state=SEED)
             for d, s in DOMAIN_MIX.items()]
    workload = pd.concat(parts)
    if len(workload) != N_REQUESTS:
        print(f"WARNING: DOMAIN_MIX gives {len(workload)} requests instead of {N_REQUESTS} "
              f"(a domain has too few requests, or the shares do not sum to 1)")
else:
    workload = candidates.sample(min(N_REQUESTS, len(candidates)), random_state=SEED)
workload = workload.reset_index(drop=True)
# a request drawn more than once (SUBJECT_FOCUS with replacement) gets a suffix: every request of the batch has its own identifier
n_copy = workload.groupby("request_id").cumcount()
workload["source_request_id"] = workload["request_id"]
workload["request_id"] = workload["request_id"].astype(str) + np.where(n_copy > 0, "#" + n_copy.astype(str), "")
rng_reference = copy.deepcopy(rng)     # same random numbers: with the reference mix, the tiers a default run would draw
workload["tier"] = rng.choice(list(TIER_MIX), size=len(workload), p=list(TIER_MIX.values()))
workload["tier_reference"] = rng_reference.choice(list(REFERENCE_TIER_MIX), size=len(workload), p=list(REFERENCE_TIER_MIX.values()))

REQUEST_IDS = list(workload["request_id"])
N = len(REQUEST_IDS)
assert len(set(REQUEST_IDS)) == N, "duplicate request identifiers"
print(f"R: {N} requests ({workload['source_request_id'].nunique()} distinct) | with a reference length (Magpie): "
      f"{int(workload['has_reference_length'].sum())}")
show(pd.crosstab(workload["domain"], workload["task_group"], margins=True), "Domain × task group")
show(workload["tier"].value_counts(), "Service tiers")


Magpie references treated as generation failures: 3
Requests too long for every model, never drawn: 0
R: 200 requests (200 distinct) | with a reference length (Magpie): 93

=== Domain × task group
task_group        code  extraction  generation  qa  reasoning  summarization  translation  All
domain                                                                                        
coding              23           0           0  10          0              0            0   33
general              0           7          52  18         25              1            0  103
humanities           0           0           0   8          4              0            3   15
mathematics          0           0           0   3         28              0            0   31
natural_sciences     0           0           0   5          0              0            0    5
social_sciences      0           0           0   5          8              0            0   13
All                 23           7         

## Request metadata

### Subjects

$S_i$: MMLU-Pro subjects of request $i$, from the Dewey code when the source has one, otherwise from the domain; general requests use all subjects.

In [7]:
T_PREPARATION_START = time.perf_counter()     # time of a decision: data preparation starts here (sections 5-11)
workload["subjects"] = workload.apply(request_subjects, axis=1)
show(workload["subjects"].map(lambda s: ", ".join(s) if s else "all").value_counts(), "Subjects used")



=== Subjects used
subjects
all                                     103
computer_science                         33
math                                     31
history, philosophy                      11
economics, psychology, law, business      5
economics                                 5
philosophy                                3
law                                       2
health                                    2
biology, chemistry, physics, health       2
psychology                                1
chemistry                                 1
history                                   1


### Complexity $h_i$

$h_i = \text{rank of the label} / (\text{number of levels} - 1)$; requests without a label get `DEFAULT_COMPLEXITY`.

In [8]:
DIFFICULTY_LEVELS = {
    "routerarena_easy_medium_hard": ["easy", "medium", "hard"],
    "magpie_very_easy_to_very_hard": ["very easy", "easy", "medium", "hard", "very hard"],
    "math_level_1_to_5": ["level 1", "level 2", "level 3", "level 4", "level 5"],
    "taco_easy_to_very_hard": ["easy", "medium", "medium hard", "hard", "very hard"],
    "longbench_easy_hard": ["easy", "hard"],
}

def complexity(row):
    levels = DIFFICULTY_LEVELS.get(row["difficulty_scale"])
    raw = str(row["difficulty_raw"]).strip().lower().replace("_", " ")
    return levels.index(raw) / (len(levels) - 1) if levels and raw in levels else np.nan

workload["complexity"] = workload.apply(complexity, axis=1)
print(f"Requests without a usable difficulty label: {int(workload['complexity'].isna().sum())} (set to {DEFAULT_COMPLEXITY})")
workload["complexity"] = workload["complexity"].fillna(DEFAULT_COMPLEXITY)
wl = workload.set_index("request_id")
show(workload[["input_tokens", "predicted_output_tokens", "evaluation_output_tokens", "complexity"]].describe().round(1),
     "Size and complexity of the workload")

Requests without a usable difficulty label: 15 (set to 0.5)

=== Size and complexity of the workload
       input_tokens  predicted_output_tokens  evaluation_output_tokens  complexity
count         200.0                    200.0                     200.0       200.0
mean          194.5                    489.9                     481.5         0.5
std           288.8                    317.4                     313.6         0.3
min             3.0                     17.0                      17.0         0.0
25%            35.0                    267.0                     267.0         0.2
50%            68.5                    474.0                     413.0         0.5
75%           223.2                    651.0                     695.0         0.8
max          2272.0                    957.0                    1405.0         1.0


## Model parameters

$t^{\mathrm{load}}_j = 28.2 + \omega_j / 8.4$ s, with $\omega_j$ the size of the weights in GB. The verbosity $v_j$ is the ML.ENERGY mean output length of model $j$ divided by that of Llama-3.1-70B, read at `VERBOSITY_BATCH` = 128.

In [9]:
g = coef["num_gpus"].astype(float)
ctx = models["context_window"].astype(float)
v = (coef["avg_output_len"] / REFERENCE_LENGTH) if USE_VERBOSITY else pd.Series(1.0, index=MODEL_IDS)
missing_w = models["weights_gb"].isna()
if missing_w.any():
    print("Weights size not on the Hub, estimated as 2 bytes per parameter:", [short(m) for m in models.index[missing_w]])
w = models["weights_gb"].fillna(2 * models["params_b"])
t_load = LOAD_FIXED_S + w / LOAD_GB_PER_S
a = pd.Series({j: 1.0 if j in ALREADY_ACTIVE else 0.0 for j in MODEL_IDS})

show(pd.DataFrame({"g_j": g, "ctx_j": ctx, "v_j": v.round(3), "w_j_GB": w.round(1),
                   "t_load_s": t_load.round(1), "a_j": a}).rename(index=short), "Model parameters")

Weights size not on the Hub, estimated as 2 bytes per parameter: ['llama3-openbiollm-70b']

=== Model parameters
                                    g_j     ctx_j    v_j  w_j_GB  t_load_s  a_j
llama3-openbiollm-70b               4.0    8192.0  1.000   140.0      44.9  0.0
llama3-med42-70b                    4.0    8192.0  1.000   141.1      45.0  0.0
llama-3.1-70b-instruct              4.0  131072.0  1.000   141.1      45.0  0.0
llama-3.1-8b-instruct               1.0  131072.0  1.127    16.1      30.1  0.0
llama-3.3-70b-instruct              4.0  131072.0  1.146   141.1      45.0  0.0
meta-llama-3-70b-instruct           4.0    8192.0  1.000   141.1      45.0  0.0
meta-llama-3-8b-instruct            1.0    8192.0  1.127    16.1      30.1  0.0
llama-3.1-nemotron-70b-instruct-hf  4.0  131072.0  1.000   141.1      45.0  0.0
openmath2-llama3.1-8b               1.0  131072.0  1.127    16.1      30.1  0.0


## Quality matrix $Q = [q_{ij}]$

### Model scores

Open LLM scores / 100; MMLU-Pro subject accuracy corrected as $(\mathrm{acc} - 0.1)/0.9$ (`QUALITY_SCALE = "absolute"`).

In [ ]:
OLL = ["oll_ifeval", "oll_bbh", "oll_math_lvl_5", "oll_gpqa", "oll_musr", "oll_mmlu_pro"]
raw_subjects = models[[c for c in models.columns if c.startswith("mmlupro_oll_")]].rename(
    columns=lambda c: c.removeprefix("mmlupro_oll_"))
fallback = 0.1 + 0.9 * models["oll_mmlu_pro"] / 100       # overall MMLU-Pro accuracy, for models without subjects
for m in raw_subjects.index[raw_subjects.isna().all(axis=1)]:
    print(f"{short(m)}: no per-subject scores, overall MMLU-Pro used for every subject")
subjects_imputed = raw_subjects.isna().sum(axis=1)                # used in the table of imputed values
raw_subjects = raw_subjects.apply(lambda col: col.fillna(fallback))

if QUALITY_SCALE == "absolute":
    scale = lambda s: s / 100
    bench = pd.DataFrame({b: scale(models[b]) for b in OLL})
    bench["reasoning_mix"] = scale(models[["oll_bbh", "oll_musr"]].mean(axis=1))
    bench["all6"] = scale(models[OLL].mean(axis=1))
    subjects_score = ((raw_subjects - 0.1) / 0.9).clip(lower=0)
elif QUALITY_SCALE == "relative":
    scale = lambda s: s / s.max()
    bench = pd.DataFrame({b: scale(models[b]) for b in OLL})
    bench["reasoning_mix"] = scale(models[["oll_bbh", "oll_musr"]].mean(axis=1))
    bench["all6"] = scale(models[OLL].mean(axis=1))
    subjects_score = raw_subjects / raw_subjects.max()
else:
    raise ValueError(QUALITY_SCALE)

show(bench.round(3).rename(index=short), f"Benchmark scores ({QUALITY_SCALE})")
show(subjects_score.round(3).rename(index=short), f"MMLU-Pro subject scores ({QUALITY_SCALE})")


=== Benchmark scores (absolute)
                                    oll_ifeval  oll_bbh  oll_math_lvl_5  oll_gpqa  oll_musr  oll_mmlu_pro  reasoning_mix   all6
model_key                                                                                                                      
llama3-openbiollm-70b                    0.760    0.471           0.197     0.097     0.143         0.430          0.307  0.350
llama3-med42-70b                         0.629    0.530           0.226     0.130     0.186         0.440          0.358  0.357
llama-3.1-70b-instruct                   0.867    0.559           0.381     0.142     0.177         0.479          0.368  0.434
llama-3.1-8b-instruct                    0.492    0.294           0.156     0.087     0.086         0.311          0.190  0.238
llama-3.3-70b-instruct                   0.900    0.566           0.483     0.105     0.156         0.481          0.361  0.448
meta-llama-3-70b-instruct                0.810    0.502           0.245

### Domain score $D_{ij}$

$$D_{ij} = \frac{1}{|S_i|} \sum_{s \in S_i} \kappa^{\mathrm{area}}_{js}$$

In [11]:
def domain_score(subj_list):
    cols = [s for s in (subj_list or subjects_score.columns) if s in subjects_score.columns]
    return subjects_score[cols].mean(axis=1) if cols else subjects_score.mean(axis=1)

D = pd.DataFrame([domain_score(s)[MODEL_IDS].values for s in workload["subjects"]], index=REQUEST_IDS, columns=MODEL_IDS)
show(D.groupby(workload["domain"].values).mean().round(3).rename(columns=short), "Mean D_ij by request domain")


=== Mean D_ij by request domain
                  llama3-openbiollm-70b  llama3-med42-70b  llama-3.1-70b-instruct  llama-3.1-8b-instruct  llama-3.3-70b-instruct  meta-llama-3-70b-instruct  meta-llama-3-8b-instruct  llama-3.1-nemotron-70b-instruct-hf  openmath2-llama3.1-8b
coding                            0.453             0.509                   0.545                  0.325                   0.539                      0.531                     0.287                               0.528                  0.079
general                           0.459             0.469                   0.509                  0.334                   0.513                      0.496                     0.309                               0.467                  0.067
humanities                        0.501             0.494                   0.559                  0.350                   0.581                      0.523                     0.295                               0.514                  0.042
mat

### Task score $T_{ij}$

Score of model $j$ on the benchmark that matches the task of request $i$.

In [12]:
TASK_BENCH = {"qa": "oll_mmlu_pro", "reasoning": "reasoning_mix", "generation": "oll_ifeval",
              "code": "subject:computer_science", "summarization": "all6", "extraction": "all6", "translation": "all6"}
TASK_BENCH_FINE = {"math_problem_solving": "oll_math_lvl_5"}

def task_score(task, group):
    key = TASK_BENCH_FINE.get(task, TASK_BENCH.get(group, "all6"))
    return domain_score([key.split(":")[1]]) if key.startswith("subject:") else bench[key]

T_score = pd.DataFrame([task_score(t, g_)[MODEL_IDS].values for t, g_ in zip(workload["task"], workload["task_group"])],
                       index=REQUEST_IDS, columns=MODEL_IDS)
show(T_score.groupby(workload["task_group"].values).mean().round(3).rename(columns=short), "Mean T_ij by task group")


=== Mean T_ij by task group
               llama3-openbiollm-70b  llama3-med42-70b  llama-3.1-70b-instruct  llama-3.1-8b-instruct  llama-3.3-70b-instruct  meta-llama-3-70b-instruct  meta-llama-3-8b-instruct  llama-3.1-nemotron-70b-instruct-hf  openmath2-llama3.1-8b
code                           0.453             0.509                   0.545                  0.325                   0.539                      0.531                     0.287                               0.528                  0.079
extraction                     0.350             0.357                   0.434                  0.238                   0.448                      0.364                     0.206                               0.369                  0.128
generation                     0.760             0.629                   0.867                  0.492                   0.900                      0.810                     0.478                               0.738                  0.233
qa                 

### Hard score $H_{ij}$

MATH Level 5 for mathematics, GPQA for natural sciences and health, BBH otherwise.

In [13]:
SCIENCE = {"biology", "chemistry", "physics", "health"}

def hard_bench(subj_list, task):
    if task == "math_problem_solving" or subj_list == ["math"]:
        return "oll_math_lvl_5"
    if subj_list and set(subj_list) <= SCIENCE:
        return "oll_gpqa"
    return "oll_bbh"

workload["hard_bench"] = [hard_bench(s, t) for s, t in zip(workload["subjects"], workload["task"])]
wl = workload.set_index("request_id")
H = pd.DataFrame([bench.loc[MODEL_IDS, b].values for b in workload["hard_bench"]], index=REQUEST_IDS, columns=MODEL_IDS)
show(workload["hard_bench"].value_counts(), "Hard benchmark used")


=== Hard benchmark used
hard_bench
oll_bbh           164
oll_math_lvl_5     31
oll_gpqa            5


### Expected quality $q_{ij}$

$$q_{ij} = (1 - \alpha h_i)\,\big(w_D D_{ij} + w_T T_{ij}\big) + \alpha h_i \, H_{ij}$$

Hypothetical specialist: on the requests of its subject, $q_{ij} = \lambda^{\mathrm{spec}} \max_{k \ne j} q_{ik}$ ($\lambda^{\mathrm{spec}}$ = `quality_ratio`); elsewhere, the quality of its donor.

In [ ]:
base = W_DOMAIN * D + W_TASK * T_score
weight_hard = ALPHA_COMPLEXITY * workload["complexity"].values[:, None]
Q = ((1 - weight_hard) * base + weight_hard * H).round(4)
if HYPOTHETICAL_SPECIALIST:
    in_subject = workload["subjects"].map(lambda s: s == [HYPOTHETICAL_SPECIALIST["subject"]]).values
    real = [j for j in MODEL_IDS if j != HS_KEY]
    Q.loc[in_subject, HS_KEY] = (HYPOTHETICAL_SPECIALIST["quality_ratio"] * Q.loc[in_subject, real].max(axis=1)).round(4)
    print(f"Hypothetical specialist: quality set to {HYPOTHETICAL_SPECIALIST['quality_ratio']} x the best real model "
          f"on {int(in_subject.sum())} '{HYPOTHETICAL_SPECIALIST['subject']}' requests")

# quality with the reference weights used only for the budget of the batch; the hypothetical specialist is left out
weight_hard_reference = REFERENCE_ALPHA_COMPLEXITY * workload["complexity"].values[:, None]
Q_reference = ((1 - weight_hard_reference) * (REFERENCE_W_DOMAIN * D + REFERENCE_W_TASK * T_score)
               + weight_hard_reference * H).round(4)
Q_reference = Q_reference[[j for j in MODEL_IDS if j != HS_KEY]]

example = REQUEST_IDS[0]
print(f"Example request {example}: domain={wl.loc[example, 'domain']}, task={wl.loc[example, 'task']}, "
      f"complexity={wl.loc[example, 'complexity']:.2f}, hard benchmark={wl.loc[example, 'hard_bench']}")
show(pd.DataFrame({"D": D.loc[example], "T": T_score.loc[example], "base": base.loc[example], "H": H.loc[example],
                   "q": Q.loc[example]}).round(3).rename(index=short), "Quality of each model on this request")
show(Q.groupby(workload["domain"].values).mean().round(3).rename(columns=short), "Mean q_ij by request domain")

Example request taco-00064: domain=coding, task=code_generation, complexity=0.75, hard benchmark=oll_bbh

=== Quality of each model on this request
                                        D      T   base      H      q
llama3-openbiollm-70b               0.453  0.453  0.453  0.471  0.460
llama3-med42-70b                    0.509  0.509  0.509  0.530  0.517
llama-3.1-70b-instruct              0.545  0.545  0.545  0.559  0.550
llama-3.1-8b-instruct               0.325  0.325  0.325  0.294  0.313
llama-3.3-70b-instruct              0.539  0.539  0.539  0.566  0.549
meta-llama-3-70b-instruct           0.531  0.531  0.531  0.502  0.520
meta-llama-3-8b-instruct            0.287  0.287  0.287  0.268  0.280
llama-3.1-nemotron-70b-instruct-hf  0.528  0.528  0.528  0.471  0.507
openmath2-llama3.1-8b               0.079  0.079  0.079  0.163  0.110

=== Mean q_ij by request domain
                  llama3-openbiollm-70b  llama3-med42-70b  llama-3.1-70b-instruct  llama-3.1-8b-instruct  llama-3.3-70b

## Tokens and serving time

$$L_{ij} = L^{\mathrm{out}}_i \, v_j, \qquad \tilde{L}_{ij} = L_{ij} + \frac{L^{\mathrm{in}}_i}{\rho}, \qquad t_{ij} = \frac{\tilde{L}_{ij}}{\theta_j}$$

In [15]:
L_in = pd.Series(workload["input_tokens"].values, index=REQUEST_IDS, dtype=float)
theta = coef["throughput_tok_s"]

def tokens_and_time(base_tokens):
    L_out = pd.DataFrame(base_tokens[:, None] * v[MODEL_IDS].values[None, :], index=REQUEST_IDS, columns=MODEL_IDS)
    L_eq = L_out.add(L_in / INPUT_OUTPUT_RATIO, axis=0)
    return L_out, L_eq, L_eq.div(theta, axis=1)

L_out, L_eq, t_serve = tokens_and_time(workload["predicted_output_tokens"].values.astype(float))          # decisions
L_out_ref, L_eq_ref, t_serve_ref = tokens_and_time(workload["evaluation_output_tokens"].values.astype(float))  # evaluation
show(pd.DataFrame({"theta_tok_s": theta.round(0), "mean_t_ij_s": t_serve.mean().round(3),
                   "total_if_all_requests_s": t_serve.sum().round(1)}).rename(index=short), f"Serving time (estimated lengths, batch {BATCH})")


=== Serving time (estimated lengths, batch 128)
                                    theta_tok_s  mean_t_ij_s  total_if_all_requests_s
llama3-openbiollm-70b                    3876.0        0.127                     25.3
llama3-med42-70b                         3876.0        0.127                     25.3
llama-3.1-70b-instruct                   3876.0        0.127                     25.3
llama-3.1-8b-instruct                    7037.0        0.079                     15.7
llama-3.3-70b-instruct                   3895.0        0.144                     28.9
meta-llama-3-70b-instruct                3876.0        0.127                     25.3
meta-llama-3-8b-instruct                 7037.0        0.079                     15.7
llama-3.1-nemotron-70b-instruct-hf       3876.0        0.127                     25.3
openmath2-llama3.1-8b                    7037.0        0.079                     15.7


## Energy

### The time line of an active model

$$\tau^{\mathrm{idle}}_j = \mathcal{T} - (1-a_j)\, t^{\mathrm{load}}_j - \sum_i t_{ij}\, x_{ij}$$

All powers are above $P^0$ = 71.8 W per GPU; $\Delta P^{\mathrm{load}}$ = 52.2 W, $\Delta P^{\mathrm{park}}$ = 49.9 W, $P^{\mathrm{srv}}_j = \varepsilon_j \theta_j / g_j$.

$$E(x, y) = \sum_j \Big[ e^{\mathrm{act}}_j + \sum_i e^{\mathrm{srv}}_{ij}\, x_{ij} + g_j\, \Delta P^{\mathrm{park}}\, \tau^{\mathrm{idle}}_j \Big] y_j$$

In [16]:
P0 = H100_NO_MODEL_W
DP_LOAD = H100_LOADING_W - P0
DP_PARK = H100_IDLE_WITH_MODEL_W - P0
P_srv = coef["j_per_out_token"] * theta / g
print(f"Accounting: {ENERGY_ACCOUNTING} | P0 {P0} W | ΔP_load {DP_LOAD:.1f} W | ΔP_park {DP_PARK:.1f} W (per GPU)")
show(P_srv.round(0).rename(index=short).rename("serving_power_per_GPU_W"), f"Serving power at batch {BATCH} (ML.ENERGY)")

Accounting: marginal | P0 71.8 W | ΔP_load 52.2 W | ΔP_park 49.9 W (per GPU)

=== Serving power at batch 128 (ML.ENERGY)
llama3-openbiollm-70b                 546.0
llama3-med42-70b                      546.0
llama-3.1-70b-instruct                546.0
llama-3.1-8b-instruct                 669.0
llama-3.3-70b-instruct                588.0
meta-llama-3-70b-instruct             546.0
meta-llama-3-8b-instruct              669.0
llama-3.1-nemotron-70b-instruct-hf    546.0
openmath2-llama3.1-8b                 669.0


### Serving energy $e^{\mathrm{srv}}_{ij}$

$$e^{\mathrm{srv}}_{ij} = g_j\big(P^{\mathrm{srv}}_j - P^0\big)\, t_{ij}$$

In [17]:
eps = coef["j_per_out_token"]

def serving_energy(L_eq_, t_):
    return L_eq_.mul(eps, axis=1) - t_.mul(g * P0, axis=1)

E_srv = serving_energy(L_eq, t_serve)
E_srv_ref = serving_energy(L_eq_ref, t_serve_ref)
assert (E_srv > 0).all().all(), "serving energy below the baseline: check the accounting"
show(E_srv.describe().T[["mean", "min", "max"]].round(1).rename(index=short), "Serving energy per request e_srv_ij (J, estimated lengths)")


=== Serving energy per request e_srv_ij (J, estimated lengths)
                                     mean   min    max
llama3-openbiollm-70b               240.4   8.4  471.1
llama3-med42-70b                    240.4   8.4  471.1
llama-3.1-70b-instruct              240.4   8.4  471.1
llama-3.1-8b-instruct                47.0   1.6   92.1
llama-3.3-70b-instruct              298.3  10.4  584.4
meta-llama-3-70b-instruct           240.4   8.4  471.1
meta-llama-3-8b-instruct             47.0   1.6   92.1
llama-3.1-nemotron-70b-instruct-hf  240.4   8.4  471.1
openmath2-llama3.1-8b                47.0   1.6   92.1


### Activation energy $e^{\mathrm{act}}_j$

$$e^{\mathrm{act}}_j = (1 - a_j)\, g_j \, \Delta P^{\mathrm{load}} \, t^{\mathrm{load}}_j$$

In [18]:
e_act = (1 - a) * g * DP_LOAD * t_load
show(e_act.round(0).rename(index=short), "e_act_j (J)")


=== e_act_j (J)
llama3-openbiollm-70b                 9368.0
llama3-med42-70b                      9396.0
llama-3.1-70b-instruct                9396.0
llama-3.1-8b-instruct                 1572.0
llama-3.3-70b-instruct                9396.0
meta-llama-3-70b-instruct             9396.0
meta-llama-3-8b-instruct              1572.0
llama-3.1-nemotron-70b-instruct-hf    9396.0
openmath2-llama3.1-8b                 1572.0


### Parking energy $e^{\mathrm{park}}_j$

$$e^{\mathrm{park}}_j = g_j \, \Delta P^{\mathrm{park}} \, \tau^{\mathrm{idle}}_j$$

In [19]:
available_time = WORKLOAD_DURATION_S - (1 - a) * t_load
assert (available_time > 0).all(), "T is shorter than a loading time"
park_full = g * DP_PARK * available_time
show(pd.DataFrame({"available_time_s": available_time.round(1), "parking_if_no_request_J": park_full.round(0)}).rename(index=short),
     f"Parking for T = {WORKLOAD_DURATION_S} s")


=== Parking for T = 600 s
                                    available_time_s  parking_if_no_request_J
llama3-openbiollm-70b                          555.1                 110805.0
llama3-med42-70b                               555.0                 110778.0
llama-3.1-70b-instruct                         555.0                 110778.0
llama-3.1-8b-instruct                          569.9                  28437.0
llama-3.3-70b-instruct                         555.0                 110778.0
meta-llama-3-70b-instruct                      555.0                 110778.0
meta-llama-3-8b-instruct                       569.9                  28437.0
llama-3.1-nemotron-70b-instruct-hf             555.0                 110778.0
openmath2-llama3.1-8b                          569.9                  28437.0


### The same energy, written for the optimizer

$$E(x, y) = \sum_{j} F_j \, y_j + \sum_{i} \sum_{j \in A_i} e'_{ij} \, x_{ij}$$

$$F_j = e^{\mathrm{act}}_j + g_j \Delta P^{\mathrm{park}} \big(\mathcal{T} - (1-a_j)\,t^{\mathrm{load}}_j\big), \qquad e'_{ij} = e^{\mathrm{srv}}_{ij} - g_j \Delta P^{\mathrm{park}} t_{ij}$$

In [20]:
F = e_act + park_full
E_net = E_srv - t_serve.mul(g * DP_PARK, axis=1)
assert (E_net > 0).all().all(), "serving costs less than parking: check powers"
show(pd.DataFrame({"F_j_kJ": (F / 1e3).round(1), "mean_e_srv_J": E_srv.mean().round(1),
                   "mean_e_net_J": E_net.mean().round(1)}).rename(index=short), "Energy parameters of the optimiser")


=== Energy parameters of the optimiser
                                    F_j_kJ  mean_e_srv_J  mean_e_net_J
llama3-openbiollm-70b                120.2         240.4         215.1
llama3-med42-70b                     120.2         240.4         215.1
llama-3.1-70b-instruct               120.2         240.4         215.1
llama-3.1-8b-instruct                 30.0          47.0          43.1
llama-3.3-70b-instruct               120.2         298.3         269.4
meta-llama-3-70b-instruct            120.2         240.4         215.1
meta-llama-3-8b-instruct              30.0          47.0          43.1
llama-3.1-nemotron-70b-instruct-hf   120.2         240.4         215.1
openmath2-llama3.1-8b                 30.0          47.0          43.1


## Monetary cost

### Prices

OpenRouter prices; a model without a provider inherits the mean price of the priced models with the same architecture.

In [21]:
fp = models["arch_fingerprint"]
p_in, p_out, price_source = {}, {}, {}
for j in MODEL_IDS:
    if pd.notna(models.loc[j, "price_out_usd_per_mtok"]):
        p_in[j], p_out[j] = models.loc[j, "price_in_usd_per_mtok"], models.loc[j, "price_out_usd_per_mtok"]
        price_source[j] = "measured (OpenRouter)"
    else:
        donors = models_full[(models_full["arch_fingerprint"] == fp[j]) & models_full["price_out_usd_per_mtok"].notna()]   # whole pool, also with EXCLUDE_MODELS
        assert len(donors), f"no priced model with the architecture of {j}"
        p_in[j], p_out[j] = donors["price_in_usd_per_mtok"].mean(), donors["price_out_usd_per_mtok"].mean()
        price_source[j] = "inherited from " + ", ".join(short(d) for d in donors.index)
prices = pd.DataFrame({"in_usd_per_Mtok": p_in, "out_usd_per_Mtok": p_out, "source": price_source})
show(prices.rename(index=short), "Market prices")


=== Market prices
                                    in_usd_per_Mtok  out_usd_per_Mtok                                                         source
llama3-openbiollm-70b                          0.25              0.36  inherited from llama-3.1-70b-instruct, llama-3.3-70b-instruct
llama3-med42-70b                               0.25              0.36  inherited from llama-3.1-70b-instruct, llama-3.3-70b-instruct
llama-3.1-70b-instruct                         0.40              0.40                                          measured (OpenRouter)
llama-3.1-8b-instruct                          0.05              0.08                                          measured (OpenRouter)
llama-3.3-70b-instruct                         0.10              0.32                                          measured (OpenRouter)
meta-llama-3-70b-instruct                      0.25              0.36  inherited from llama-3.1-70b-instruct, llama-3.3-70b-instruct
meta-llama-3-8b-instruct                       0.0

### Cost of a solution

$$C(x) = \sum_{i} \sum_{j \in A_i} c^{\mathrm{srv}}_{ij}\, x_{ij}, \qquad c^{\mathrm{srv}}_{ij} = L^{\mathrm{in}}_i \, p^{\mathrm{in}}_j + L_{ij} \, p^{\mathrm{out}}_j$$

In [22]:
def token_cost(L_out_):
    return L_out_.mul(pd.Series(p_out)[MODEL_IDS] / 1e6, axis=1).add(
        np.outer(L_in.values, pd.Series(p_in)[MODEL_IDS].values / 1e6), fill_value=0)

if COST_MODE == "market":
    c, c_ref = token_cost(L_out), token_cost(L_out_ref)
    c_fix = pd.Series(0.0, index=MODEL_IDS)
elif COST_MODE == "gpu_rental":
    c = c_ref = pd.DataFrame(0.0, index=REQUEST_IDS, columns=MODEL_IDS)
    c_fix = g * H100_USD_PER_HOUR * WORKLOAD_DURATION_S / 3600
else:
    raise ValueError(COST_MODE)
show(pd.DataFrame({"mean_c_ij_usd": c.mean().round(6), "c_fix_j_usd": c_fix.round(4)}).rename(index=short),
     f"Cost parameters ({COST_MODE}, estimated lengths)")


=== Cost parameters (market, estimated lengths)
                                    mean_c_ij_usd  c_fix_j_usd
llama3-openbiollm-70b                    0.000225          0.0
llama3-med42-70b                         0.000225          0.0
llama-3.1-70b-instruct                   0.000274          0.0
llama-3.1-8b-instruct                    0.000054          0.0
llama-3.3-70b-instruct                   0.000199          0.0
meta-llama-3-70b-instruct                0.000225          0.0
meta-llama-3-8b-instruct                 0.000054          0.0
llama-3.1-nemotron-70b-instruct-hf       0.000225          0.0
openmath2-llama3.1-8b                    0.000054          0.0


### Checks on the inputs and imputed values

In [23]:
price_df = pd.DataFrame({"in": pd.Series(p_in), "out": pd.Series(p_out)})
assert price_df.notna().all().all() and (price_df >= 0).all().all(), "missing or negative price"
for name, col in {"energy per output token": coef["j_per_out_token"], "throughput": theta,
                  "context window": ctx, "number of GPUs": g, "loading time": t_load}.items():
    assert col.notna().all() and (col > 0).all(), f"missing or non-positive {name}"
assert np.isfinite(F).all() and np.isfinite(E_net.values).all() and np.isfinite(c.values).all(), "non-finite coefficient"
print("Input checks passed: prices, energy per token, throughput, context window, GPUs, loading time.")

imputed = pd.DataFrame({
    "energy": [("measured" if models.loc[j, "mle_link"] == "direct" else f"inherited from {short(models.loc[j, 'mle_donor'])}") for j in MODEL_IDS],
    "price": [price_source[j] for j in MODEL_IDS],
    "weights_GB": ["estimated (2 bytes per parameter)" if missing_w[j] else "safetensors metadata" for j in MODEL_IDS],
    "MMLU-Pro subjects": [("all measured" if subjects_imputed[j] == 0 else f"{int(subjects_imputed[j])} replaced by the overall score") for j in MODEL_IDS],
}, index=MODEL_IDS)
show(imputed.rename(index=short), "Measured, imputed or inherited values")

Input checks passed: prices, energy per token, throughput, context window, GPUs, loading time.

=== Measured, imputed or inherited values
                                                                   energy                                                          price                         weights_GB MMLU-Pro subjects
llama3-openbiollm-70b               inherited from llama-3.1-70b-instruct  inherited from llama-3.1-70b-instruct, llama-3.3-70b-instruct  estimated (2 bytes per parameter)      all measured
llama3-med42-70b                    inherited from llama-3.1-70b-instruct  inherited from llama-3.1-70b-instruct, llama-3.3-70b-instruct               safetensors metadata      all measured
llama-3.1-70b-instruct                                           measured                                          measured (OpenRouter)               safetensors metadata      all measured
llama-3.1-8b-instruct                                            measured                             

## Eligible models $A_i$

$$\Phi_i = \{ j : L^{\mathrm{in}}_i + L_{ij} \le \mathrm{ctx}_j \}, \qquad q^{\min}_i = \gamma_{s_i} \max_{k \in \Phi_i} q_{ik}, \qquad A_i = \{\, j \in \Phi_i : q_{ij} \ge q^{\min}_i \,\}$$

with $\gamma_s$ = `TIER_RATIO`.

In [ ]:
fits = L_out.add(L_in, axis=0).le(ctx, axis=1)
best_q = Q.where(fits).max(axis=1)
q_min = pd.Series(workload["tier"].map(TIER_RATIO).values * best_q.values, index=REQUEST_IDS)
eligible = Q.ge(q_min, axis=0) & fits
A = {i: [j for j in MODEL_IDS if eligible.loc[i, j]] for i in REQUEST_IDS}
assert all(A[i] for i in REQUEST_IDS), "a request has no eligible model"
n_eligible_requests = eligible.sum()      # used to break ties in the baselines
# eligibility in the reference configuration: reference tiers, ratios and quality, pool without the hypothetical specialist
REFERENCE_MODELS = list(Q_reference.columns)
fits_reference = fits[REFERENCE_MODELS]
best_q_reference = Q_reference.where(fits_reference).max(axis=1)
q_min_reference = pd.Series(workload["tier_reference"].map(REFERENCE_TIER_RATIO).values * best_q_reference.values, index=REQUEST_IDS)
eligible_reference = Q_reference.ge(q_min_reference, axis=0) & fits_reference
assert eligible_reference.any(axis=1).all(), "a request has no eligible model in the reference configuration"
PREPARATION_SECONDS = time.perf_counter() - T_PREPARATION_START

print(f"Requests too long for at least one model: {int((~fits).any(axis=1).sum())} of {N}")
show(pd.DataFrame({"too_long_for_context": (~fits).sum(), "eligible_for": n_eligible_requests}).rename(index=short),
     "Number of requests per model")
show(eligible.sum(axis=1).groupby(workload["tier"].values).describe()[["mean", "min", "max"]],
     "Eligible models per request, by tier")
relative = Q.div(best_q, axis=0).where(fits)
show(relative.describe(percentiles=[0.1, 0.5, 0.9]).T[["10%", "50%", "90%"]].round(2).rename(index=short),
     "Quality relative to the best model of each request (to calibrate TIER_RATIO)")

Requests too long for at least one model: 0 of 200

=== Number of requests per model
                                    too_long_for_context  eligible_for
llama3-openbiollm-70b                                  0           148
llama3-med42-70b                                       0           148
llama-3.1-70b-instruct                                 0           194
llama-3.1-8b-instruct                                  0            38
llama-3.3-70b-instruct                                 0           200
meta-llama-3-70b-instruct                              0           151
meta-llama-3-8b-instruct                               0            37
llama-3.1-nemotron-70b-instruct-hf                     0           164
openmath2-llama3.1-8b                                  0             0

=== Eligible models per request, by tier
              mean  min  max
Economy   7.744186  6.0  8.0
Premium   1.914286  1.0  3.0
Standard  5.573770  3.0  6.0

=== Quality relative to the best model of each

### Budget $B$

$$C_{\min} = \sum_{i \in R} \min_{j \in A_i} c^{\mathrm{srv}}_{ij}, \qquad B = (1 + \delta)\, C_{\min}$$

$C_{\min}$ is computed with the default parameters (`REFERENCE_*`) and the pool of the provider, so every run on the same batch has the same budget; $\delta$ = `BUDGET_MARGIN`. `CHEAPEST_COST_USD` is the same sum with the parameters of the run, only reported.

**Capacity pre-check.** If the GPU time the batch needs at least exceeds the time all models together can offer, the run is marked infeasible without calling the solver.

In [ ]:
assert COST_MODE == "market", "the closed form of the budget assumes prices per token (no fixed cost per model)"
t0 = time.perf_counter()
# capacity pre-check: GPU time needed at least (each request on its fastest eligible model) vs time offered by all models
NEEDED_TIME_S = float(sum(min(t_serve.loc[i, j] for j in A[i]) for i in REQUEST_IDS))
OFFERED_TIME_S = float(available_time[MODEL_IDS].sum())
CAPACITY_PROVEN_INFEASIBLE = bool(ENFORCE_CAPACITY and NEEDED_TIME_S > OFFERED_TIME_S)
print(f"Capacity pre-check: GPU time needed at least {NEEDED_TIME_S:.1f} s | offered by all models {OFFERED_TIME_S:.1f} s"
      + (" -> no assignment respects capacity" if CAPACITY_PROVEN_INFEASIBLE else ""))

# budget of the batch: cheapest eligible model of every request in the reference configuration
C_min = float(c[REFERENCE_MODELS].where(eligible_reference).min(axis=1).sum())
B = (1 + BUDGET_MARGIN) * C_min
CHEAPEST_COST_USD = float(c.where(eligible).min(axis=1).sum())      # same sum with the parameters of this run (information only)
C_max = float(c.where(eligible).max(axis=1).sum())                  # every request on its most expensive eligible model (information only)
BUDGET_SECONDS = time.perf_counter() - t0
print(f"C_min_ref (reference configuration) = {C_min:.4f} USD | budget B = {B:.4f} USD (delta {BUDGET_MARGIN:.0%})")
print(f"Cheapest cost with the parameters of this run = {CHEAPEST_COST_USD:.4f} USD ({(B / CHEAPEST_COST_USD - 1):+.1%} room) | "
      f"most expensive eligible = {C_max:.4f} USD")
if B < CHEAPEST_COST_USD - 1e-6:
    print("The budget is below the cheapest cost of this run: no solution respects it ('Solve' names the constraint and goes on).")


Capacity pre-check: GPU time needed at least 24.0 s | offered by all models 5039.8 s
C_min_ref (reference configuration) = 0.0351 USD | budget B = 0.0421 USD (delta 20%)
Cheapest cost with the parameters of this run = 0.0351 USD (+20.0% room) | most expensive eligible = 0.0544 USD


## Optimization model (FL-MILP)

$$
\begin{aligned}
\min_{x,y} \quad & \sum_{j \in M} F_j\, y_j + \sum_{i \in R} \sum_{j \in A_i} e'_{ij}\, x_{ij}\\
\text{s.t.} \quad & \sum_{j \in A_i} x_{ij} = 1 && \forall i \in R\\
& x_{ij} \le y_j && \forall i \in R,\ j \in A_i\\
& \sum_{i}\sum_{j \in A_i} c^{\mathrm{srv}}_{ij}\, x_{ij} \le B\\
& \sum_{i : j \in A_i} t_{ij}\, x_{ij} \le \big(\mathcal{T} - (1-a_j)\, t^{\mathrm{load}}_j\big)\, y_j && \forall j \in M\\
& x_{ij},\, y_j \in \{0, 1\}
\end{aligned}
$$

### Create the problem

In [ ]:
# the sections below define one small function per step and apply it at once
# same functions are used by the synthetic test so the test checks exactly the code that builds this MILP
def create_problem(name):
    return pulp.LpProblem(name, pulp.LpMinimize)

T_MILP_START = time.perf_counter()      # time of a decision: building the MILP starts here
prob = create_problem("fl_milp")

### Decision variables

In [27]:
def add_variables(models_, requests_, A_):
    y_ = {j: pulp.LpVariable(f"y_{k}", cat="Binary") for k, j in enumerate(models_)}
    pairs_ = [(i, j) for i in requests_ for j in A_[i]]
    x_ = {(i, j): pulp.LpVariable(f"x_{n}", cat="Binary") for n, (i, j) in enumerate(pairs_)}
    return y_, x_, pairs_

y, x, pairs = add_variables(MODEL_IDS, REQUEST_IDS, A)
print(f"Selection variables y_j: {len(y)} | assignment variables x_ij: {len(x)} "
      f"(out of {N * K} possible pairs, {N * K - len(x)} removed by eligibility)")

Selection variables y_j: 9 | assignment variables x_ij: 1080 (out of 1800 possible pairs, 720 removed by eligibility)


### Objective function: total energy

In [28]:
def add_objective(prob_, y_, x_, pairs_, F_, E_net_):
    prob_ += pulp.lpSum(F_[j] * y_[j] for j in y_) + pulp.lpSum(E_net_.loc[i, j] * x_[(i, j)] for i, j in pairs_), "total_energy"

add_objective(prob, y, x, pairs, F, E_net)

### Constraint 1: every request is served by exactly one model

In [29]:
def add_assignment(prob_, x_, requests_, A_):
    for n, i in enumerate(requests_):
        prob_ += pulp.lpSum(x_[(i, j)] for j in A_[i]) == 1, f"assign_{n}"

add_assignment(prob, x, REQUEST_IDS, A)
print(f"{N} assignment constraints")

200 assignment constraints


### Constraint 2: a request can go only to an active model

In [30]:
def add_linking(prob_, x_, y_, pairs_):
    for n, (i, j) in enumerate(pairs_):
        prob_ += x_[(i, j)] <= y_[j], f"link_{n}"

add_linking(prob, x, y, pairs)
print(f"{len(pairs)} linking constraints")

1080 linking constraints


### Constraint 3: budget

In [31]:
def add_budget(prob_, x_, y_, pairs_, c_, c_fix_, B_):
    prob_ += (pulp.lpSum(c_.loc[i, j] * x_[(i, j)] for i, j in pairs_)
              + pulp.lpSum(c_fix_[j] * y_[j] for j in y_)) <= B_, "budget"

add_budget(prob, x, y, pairs, c, c_fix, B)
print(f"Budget constraint: total cost ≤ {B:.4f} USD")

Budget constraint: total cost ≤ 0.0421 USD


### Constraint 4: capacity in the period $\mathcal{T}$

In [32]:
def add_capacity(prob_, x_, y_, pairs_, t_, available_):
    for k, j in enumerate(y_):
        prob_ += (pulp.lpSum(t_.loc[i, jj] * x_[(i, jj)] for i, jj in pairs_ if jj == j)
                  <= available_[j] * y_[j]), f"capacity_{k}"

if ENFORCE_CAPACITY:
    add_capacity(prob, x, y, pairs, t_serve, available_time)
    print(f"{K} capacity constraints")
else:
    print("Capacity not enforced")

9 capacity constraints


### Solve

CBC with a time limit of `SOLVER_TIME_LIMIT_S`; a solution stopped by the limit is marked as not proven optimal. If the problem is infeasible, it is solved again without the budget and without each capacity constraint in turn, to find the cause, and the relaxed solution is marked as infeasible.

In [ ]:
def solve(prob_, n_models, capacity_infeasible=False):
    # returns the solver status and, if the problem is infeasible, the constraints whose removal makes it feasible
    if capacity_infeasible:              # proven by the pre-check of budget: no assignment respects capacity
        return "Infeasible", [f"capacity_{k}" for k in range(n_models) if f"capacity_{k}" in prob_.constraints]
    status_ = pulp.LpStatus[prob_.solve(pulp.PULP_CBC_CMD(msg=False, timeLimit=SOLVER_TIME_LIMIT_S))]
    culprits = []
    if status_ != "Optimal":
        for name in ["budget"] + [f"capacity_{k}" for k in range(n_models)]:
            if name in prob_.constraints:
                relaxed = prob_.deepcopy()
                del relaxed.constraints[name]
                if pulp.LpStatus[relaxed.solve(pulp.PULP_CBC_CMD(msg=False, timeLimit=DIAGNOSIS_TIME_LIMIT_S))] == "Optimal":
                    culprits.append(name)
    return status_, culprits

MILP_BUILD_SECONDS = time.perf_counter() - T_MILP_START
t0 = time.perf_counter()
status, culprits = solve(prob, K, CAPACITY_PROVEN_INFEASIBLE)
MILP_SOLVE_SECONDS = time.perf_counter() - t0
print("Solver status:", status)
MILP_STATUS = "optimal"
if status != "Optimal":
    if CAPACITY_PROVEN_INFEASIBLE:
        print("Capacity infeasible (pre-check, 'Budget'): solved without the capacity constraints.")
    else:
        for name in culprits:
            print(f"Feasible without constraint '{name}': this constraint makes the problem infeasible.")
    drop = culprits or [n for n in ["budget"] + [f"capacity_{k}" for k in range(K)] if n in prob.constraints]
    for name in drop:
        del prob.constraints[name]
    status_relaxed = pulp.LpStatus[prob.solve(pulp.PULP_CBC_CMD(msg=False, timeLimit=SOLVER_TIME_LIMIT_S))]
    assert status_relaxed == "Optimal", "no solution even without budget and capacity"
    MILP_STATUS = "infeasible (solved without: " + ", ".join(sorted({n.split("_")[0] for n in drop})) + ")"
    print(f"No feasible solution. {MILP_STATUS}: the solution below violates those constraints.")
elif prob.sol_status != 1:                # stopped at the time limit (600 s) with a solution not proven optimal
    MILP_STATUS = "feasible, not proven optimal (time limit)"
print(f"Total energy of the MILP solution (estimated lengths): {pulp.value(prob.objective) / 1e3:.3f} kJ | status: {MILP_STATUS}")

Solver status: Optimal
Total energy of the MILP solution (estimated lengths): 174.057 kJ | status: optimal


## Results of the optimization

### Selection $M^*$ and batch diagnostic

In [34]:
M_star = [j for j in MODEL_IDS if y[j].value() > 0.5]
milp_assign = pd.Series({i: j for (i, j) in pairs if x[(i, j)].value() > 0.5}).reindex(REQUEST_IDS)
n_assigned = pd.Series({j: int((milp_assign == j).sum()) for j in MODEL_IDS})

selection = pd.DataFrame({
    "y_j": [int(j in M_star) for j in MODEL_IDS],
    "role": models.loc[MODEL_IDS, "role"].values,
    "gpus": g[MODEL_IDS].values,
    "requests_assigned": n_assigned.values,
    "eligible_for": n_eligible_requests[MODEL_IDS].values,
    "below_batch": [bool(j in M_star and n_assigned[j] < BATCH) for j in MODEL_IDS],
}, index=[short(j) for j in MODEL_IDS])
print("M* =", [short(j) for j in M_star])
show(selection, "Selection")
weak = [short(j) for j in M_star if n_assigned[j] < BATCH]
if weak:
    print(f"Active models with fewer than {BATCH} requests (batch assumption weakest): {weak}")

M* = ['llama-3.3-70b-instruct']

=== Selection
                                    y_j         role  gpus  requests_assigned  eligible_for  below_batch
llama3-openbiollm-70b                 0     medicine   4.0                  0           148        False
llama3-med42-70b                      0     medicine   4.0                  0           148        False
llama-3.1-70b-instruct                0   generalist   4.0                  0           194        False
llama-3.1-8b-instruct                 0   generalist   1.0                  0            38        False
llama-3.3-70b-instruct                1   generalist   4.0                200           200        False
meta-llama-3-70b-instruct             0   generalist   4.0                  0           151        False
meta-llama-3-8b-instruct              0   generalist   1.0                  0            37        False
llama-3.1-nemotron-70b-instruct-hf    0   generalist   4.0                  0           164        False
openmath

### Assignment $r_i \rightarrow m_j$

In [35]:
rows = []
for i, j in milp_assign.items():
    alts = [k for k in A[i] if k != j]
    alt = min(alts, key=lambda k: E_net.loc[i, k]) if alts else None
    if alt is None:
        why = "only eligible model"
    elif alt not in M_star:
        why = "alternative not active: activating it costs more energy"
    elif E_net.loc[i, j] <= E_net.loc[i, alt]:
        why = "lowest energy among the active models"
    else:
        why = "budget or capacity"
    rows.append({
        "request": i, "domain": wl.loc[i, "domain"], "task": wl.loc[i, "task"], "tier": wl.loc[i, "tier"],
        "c_i": round(wl.loc[i, "complexity"], 2), "in_tok": int(L_in[i]),
        "out_tok_est": int(wl.loc[i, "predicted_output_tokens"]), "out_tok_ref": int(wl.loc[i, "evaluation_output_tokens"]),
        "q_min": round(q_min[i], 3), "model": short(j), "q_ij": round(Q.loc[i, j], 3), "best_q": round(best_q[i], 3),
        "e_srv_J": round(E_srv.loc[i, j], 1), "c_ij_usd": round(c.loc[i, j], 6),
        "n_eligible": len(A[i]), "closest_alternative": short(alt) if alt else "-", "why_not_alternative": why,
    })
assignment = pd.DataFrame(rows).set_index("request")
show(assignment.head(50), "Assignment (first 50 requests; the full table is saved in 'Save')")


=== Assignment (first 50 requests; the full table is saved in section 17)
                            domain                  task      tier   c_i  in_tok  out_tok_est  out_tok_ref  q_min                   model   q_ij  best_q  e_srv_J  c_ij_usd  n_eligible     closest_alternative                                      why_not_alternative
request                                                                                                                                                                                                                                                                 
taco-00064                  coding       code_generation  Standard  0.75     512          957          957  0.413  llama-3.3-70b-instruct  0.549   0.550    583.0  0.000402           6   llama3-openbiollm-70b  alternative not active: activating it costs more energy
routerarena-01552       humanities           translation  Standard  1.00      35          151          151  0.406  llama-3.3-70b-i

In [36]:
show(pd.crosstab(assignment["model"], assignment["tier"], margins=True), "Requests per model and tier")
show(pd.crosstab(assignment["model"], wl.loc[assignment.index, "domain"], margins=True), "Requests per model and domain")
show(pd.crosstab(assignment["model"], wl.loc[assignment.index, "task_group"], margins=True), "Requests per model and task group")
show(assignment["why_not_alternative"].value_counts(), "Why each request went to its model")


=== Requests per model and tier
tier                    Economy  Premium  Standard  All
model                                                  
llama-3.3-70b-instruct       43       35       122  200
All                          43       35       122  200

=== Requests per model and domain
domain                  coding  general  humanities  mathematics  natural_sciences  social_sciences  All
model                                                                                                   
llama-3.3-70b-instruct      33      103          15           31                 5               13  200
All                         33      103          15           31                 5               13  200

=== Requests per model and task group
task_group              code  extraction  generation  qa  reasoning  summarization  translation  All
model                                                                                               
llama-3.3-70b-instruct    23           7       

### Time line and energy balance (estimated lengths)

The energy recomputed from the time line must equal the objective of the MILP.

In [37]:
def timeline(assign, t_, E_srv_, c_):
    rows = {}
    for j in sorted(set(assign.values)):
        served = assign.index[assign == j]
        t_s = t_.loc[served, j].sum()
        t_l = (1 - a[j]) * t_load[j]
        t_idle = WORKLOAD_DURATION_S - t_l - t_s
        rows[short(j)] = {
            "requests": len(served), "loading_s": t_l, "serving_s": t_s, "parked_s": t_idle,
            "activation_kJ": e_act[j] / 1e3,
            "serving_kJ": E_srv_.loc[served, j].sum() / 1e3,
            "parking_kJ": g[j] * DP_PARK * max(t_idle, 0) / 1e3,
            "cost_usd": c_.loc[served, j].sum() + c_fix[j],
        }
    tl = pd.DataFrame(rows).T
    tl["total_kJ"] = tl[["activation_kJ", "serving_kJ", "parking_kJ"]].sum(axis=1)
    return tl

milp_timeline = timeline(milp_assign, t_serve, E_srv, c)
show(milp_timeline.round(3), f"Time line of the active models (T = {WORKLOAD_DURATION_S} s, estimated lengths)")
print(f"Total energy: {milp_timeline['total_kJ'].sum():.3f} kJ (objective {pulp.value(prob.objective) / 1e3:.3f} kJ)")
if "capacity" not in MILP_STATUS:       # without capacity the parked time can be negative and is counted as 0
    assert abs(milp_timeline["total_kJ"].sum() - pulp.value(prob.objective) / 1e3) < 1e-3, "energy balance does not match the objective"


=== Time line of the active models (T = 600 s, estimated lengths)
                        requests  loading_s  serving_s  parked_s  activation_kJ  serving_kJ  parking_kJ  cost_usd  total_kJ
llama-3.3-70b-instruct     200.0     44.999     28.899   526.102          9.396      59.652      105.01      0.04   174.057
Total energy: 174.057 kJ (objective 174.057 kJ)


### Evaluation with the reference lengths

The same assignment recomputed with the reference lengths of the Magpie requests; the other requests keep their estimate.

In [38]:
milp_timeline_ref = timeline(milp_assign, t_serve_ref, E_srv_ref, c_ref)
show(milp_timeline_ref.round(3), "Time line with reference lengths (Magpie) and estimates (other sources)")
e_dec, e_ref = milp_timeline["total_kJ"].sum(), milp_timeline_ref["total_kJ"].sum()
n_ref = int(wl["has_reference_length"].sum())
print(f"Requests with a reference length (Magpie): {n_ref} of {N}; the others keep their estimate")
print(f"Energy: estimated {e_dec:.3f} kJ | with reference lengths {e_ref:.3f} kJ | difference {e_ref - e_dec:+.3f} kJ ({(e_ref / e_dec - 1) * 100:+.1f}%), due to the {n_ref} Magpie requests only")
print(f"Cost: estimated {milp_timeline['cost_usd'].sum():.4f} | with reference lengths {milp_timeline_ref['cost_usd'].sum():.4f} USD (B = {B:.4f})")
print("Time feasibility with reference lengths:", bool((milp_timeline_ref["parked_s"] >= -1e-6).all()))


=== Time line with reference lengths (Magpie) and estimates (other sources)
                        requests  loading_s  serving_s  parked_s  activation_kJ  serving_kJ  parking_kJ  cost_usd  total_kJ
llama-3.3-70b-instruct     200.0     44.999     28.404   526.597          9.396       58.63     105.109     0.039   173.135
Requests with a reference length (Magpie): 93 of 200; the others keep their estimate
Energy: estimated 174.057 kJ | with reference lengths 173.135 kJ | difference -0.923 kJ (-0.5%), due to the 93 Magpie requests only
Cost: estimated 0.0398 | with reference lengths 0.0392 USD (B = 0.0421)
Time feasibility with reference lengths: True


### Diagnostics: which constraint shapes the selection

In [ ]:
# 1 every model alone
alone = pd.DataFrame({"not_eligible_requests": (~eligible).sum(),
                      "gpu_time_needed_s": t_serve[MODEL_IDS].sum(),
                      "gpu_time_available_s": available_time[MODEL_IDS],
                      "cost_usd": c[MODEL_IDS].sum() + c_fix[MODEL_IDS]}, index=MODEL_IDS)
alone["quality_ok"] = alone["not_eligible_requests"] == 0
alone["capacity_ok"] = (alone["gpu_time_needed_s"] <= alone["gpu_time_available_s"]) | (not ENFORCE_CAPACITY)
alone["budget_ok"] = alone["cost_usd"] <= B + 1e-6
show(alone.sort_values("not_eligible_requests").round(4).rename(index=short),
     f"Every model alone: which constraint it would violate (B = {B:.4f} USD)")

# 2 active models: who can serve what
if len(M_star) > 1:
    pattern = eligible[M_star].rename(columns=short).apply(lambda r: " + ".join(k for k in r.index if r[k]) or "none", axis=1)
    show(pd.crosstab([workload["domain"].values, workload["tier"].values], pattern.values,
                     rownames=["domain", "tier"], colnames=["eligible active models"]),
         "Eligible active models, by domain and tier")
    only = {short(j): int((eligible[M_star].sum(axis=1).eq(1) & eligible[j]).sum()) for j in M_star}
    print("Requests that only one active model can serve:", only)
    show(pd.DataFrame({"mean_q": Q[M_star].mean(), "mean_serving_J": E_srv[M_star].mean(),
                       "price_out_usd_per_Mtok": pd.Series(p_out)[M_star], "requests_assigned": n_assigned[M_star]}
                      ).round(4).rename(index=short), "Active models: quality, serving energy, price")

# 3 specialists on the requests of their own subject
SUBJECT_OF_ROLE = {"mathematics": "math", "medicine": "health"}
spec_rows = []
for j in MODEL_IDS:
    subj = HYPOTHETICAL_SPECIALIST["subject"] if j == HS_KEY else SUBJECT_OF_ROLE.get(models.loc[j, "role"])
    if subj is None:
        continue
    own = workload["subjects"].map(lambda s: s == [subj]).values
    row = {"model": short(j), "subject": subj, "requests_of_subject": int(own.sum())}
    if own.any():
        ratio = Q.loc[own, j] / best_q[own]
        row.update(q_over_best_median=ratio.median(), q_over_best_max=ratio.max(),
                   eligible_for=int(eligible.loc[own, j].sum()), assigned=int((milp_assign[own] == j).sum()))
    spec_rows.append(row)
if spec_rows:
    show(pd.DataFrame(spec_rows).set_index("model").round(3),
         f"Specialists on the requests of their own subject (eligible in a tier if q / best >= {TIER_RATIO})")



=== Every model alone: which constraint it would violate (B = 0.0421 USD)
                                    not_eligible_requests  gpu_time_needed_s  gpu_time_available_s  cost_usd  quality_ok  capacity_ok  budget_ok
llama-3.3-70b-instruct                                  0            28.8991              555.0015    0.0398        True         True       True
llama-3.1-70b-instruct                                  6            25.3461              555.0015    0.0548       False         True      False
llama-3.1-nemotron-70b-instruct-hf                     36            25.3461              555.0015    0.0450       False         True      False
meta-llama-3-70b-instruct                              49            25.3461              555.0015    0.0450       False         True      False
llama3-openbiollm-70b                                  52            25.3461              555.1333    0.0450       False         True      False
llama3-med42-70b                                       

## Baselines

### Evaluation of any solution

A solution is feasible if every request goes to an eligible model (quality), its cost is at most $B$ (budget) and loading and serving fit in $\mathcal{T}$ for every active model (capacity).

In [40]:
_model_col = {j: k for k, j in enumerate(MODEL_IDS)}

def not_eligible_count(assign):
    rows = eligible.index.get_indexer(assign.index)
    cols = np.array([_model_col[j] for j in assign.values])
    return int((~eligible.values[rows, cols]).sum())

def evaluate(assign, name):
    tl, tl_ref = timeline(assign, t_serve, E_srv, c), timeline(assign, t_serve_ref, E_srv_ref, c_ref)
    cost, cost_ref = tl["cost_usd"].sum(), tl_ref["cost_usd"].sum()
    not_eligible = not_eligible_count(assign)
    capacity_ok = bool((tl["parked_s"] >= -1e-6).all()) or not ENFORCE_CAPACITY
    return {
        "method": name,
        "total_energy_kJ": tl["total_kJ"].sum(),
        "activation_kJ": tl["activation_kJ"].sum(), "serving_kJ": tl["serving_kJ"].sum(), "parking_kJ": tl["parking_kJ"].sum(),
        "cost_usd": cost,
        "mean_quality": float(np.mean([Q.loc[i, j] for i, j in assign.items()])),
        "models": len(tl), "gpus": int(sum(g[j] for j in set(assign.values))),
        "to_specialists": float((models.loc[list(assign.values), "role"] != "generalist").mean()),
        "quality_ok": not_eligible == 0, "not_eligible_requests": not_eligible,
        "budget_ok": bool(cost <= B + 1e-6),      # 1e-6 USD: numerical tolerance of the solver on the budget row
        "capacity_ok": capacity_ok,
        "serving_s": float(tl["serving_s"].sum()),
        "max_utilisation": float(((tl["loading_s"] + tl["serving_s"]) / WORKLOAD_DURATION_S).max()),
        "energy_reference_length_kJ": tl_ref["total_kJ"].sum(), "cost_reference_length_usd": cost_ref,
        "feasible_reference_length": bool(not_eligible == 0 and cost_ref <= B + 1e-6
                                          and ((tl_ref["parked_s"] >= -1e-6).all() or not ENFORCE_CAPACITY)),
        "active": ", ".join(tl.index),
    }

results = {"Facility-location MILP": evaluate(milp_assign, "Facility-location MILP")}
DECISION_SECONDS = {"Facility-location MILP": MILP_BUILD_SECONDS + MILP_SOLVE_SECONDS}   # time of a decision, per method


### Quality-First

$j_i = \arg\max_{j \in A_i} q_{ij}$

In [41]:
t0 = time.perf_counter()
qf = pd.Series({i: max(A[i], key=lambda j: Q.loc[i, j]) for i in REQUEST_IDS})
DECISION_SECONDS["Quality-First"] = time.perf_counter() - t0
results["Quality-First"] = evaluate(qf, "Quality-First")

### Cost-First

$j_i = \arg\min_{j \in A_i} c^{\mathrm{srv}}_{ij}$; ties are broken by the model eligible for the most requests, then by quality.

In [42]:
t0 = time.perf_counter()
cost_key = (lambda i, j: (round(c.loc[i, j], 12), -n_eligible_requests[j], -Q.loc[i, j])) if COST_MODE == "market" \
           else (lambda i, j: (round(c_fix[j], 9), -n_eligible_requests[j], -Q.loc[i, j]))
cf = pd.Series({i: min(A[i], key=lambda j: cost_key(i, j)) for i in REQUEST_IDS})
DECISION_SECONDS["Cost-First"] = time.perf_counter() - t0
results["Cost-First"] = evaluate(cf, "Cost-First")

### Energy-First (per request)

$j_i = \arg\min_{j \in A_i} e^{\mathrm{srv}}_{ij}$, ties broken as in Cost-First.

In [43]:
t0 = time.perf_counter()
ef = pd.Series({i: min(A[i], key=lambda j: (round(E_srv.loc[i, j], 9), -n_eligible_requests[j], -Q.loc[i, j])) for i in REQUEST_IDS})
DECISION_SECONDS["Energy-First (per request)"] = time.perf_counter() - t0
results["Energy-First (per request)"] = evaluate(ef, "Energy-First (per request)")

### Random

An eligible model chosen uniformly for each request, 300 draws.

In [44]:
t0 = time.perf_counter()
draws = [pd.Series({i: A[i][rng.integers(len(A[i]))] for i in REQUEST_IDS}) for _ in range(300)]
DECISION_SECONDS["Random (mean of 300)"] = (time.perf_counter() - t0) / 300      # one draw
runs = pd.DataFrame([evaluate(d, "Random") for d in draws])
runs["feasible"] = runs["quality_ok"] & runs["budget_ok"] & runs["capacity_ok"]
spread = runs[["total_energy_kJ", "cost_usd", "mean_quality", "models", "gpus"]].agg(["mean", "std", "min", "max"]).T
show(spread.round(3), "Random: distribution over 300 runs")
print(f"Feasible runs: {runs['feasible'].mean():.0%} (quality {runs['quality_ok'].mean():.0%}, budget {runs['budget_ok'].mean():.0%}, capacity {runs['capacity_ok'].mean():.0%})")
rnd = runs.drop(columns=["method", "active", "feasible"]).mean().to_dict()
rnd.update(method="Random (mean of 300)", active="-")
results["Random (mean of 300)"] = rnd


=== Random: distribution over 300 runs
                    mean    std      min      max
total_energy_kJ  825.390  2.975  795.753  827.182
cost_usd           0.044  0.001    0.042    0.046
mean_quality       0.491  0.004    0.480    0.501
models             7.990  0.100    7.000    8.000
gpus              25.990  0.100   25.000   26.000
Feasible runs: 0% (quality 100%, budget 0%, capacity 100%)


### Single model

The feasible model with the lowest energy; if none is feasible, the model with the fewest non-eligible requests, then the lowest energy.

In [45]:
t0 = time.perf_counter()
singles = {j: evaluate(pd.Series(j, index=REQUEST_IDS), f"Single model: {short(j)}") for j in MODEL_IDS}
all_ok = [r for r in singles.values() if r["quality_ok"] and r["budget_ok"] and r["capacity_ok"]]
quality_ok_only = [r for r in singles.values() if r["quality_ok"]]
if all_ok:
    best_single = min(all_ok, key=lambda r: r["total_energy_kJ"])
    single_rule = "lowest energy among the models that respect every constraint"
elif quality_ok_only:
    best_single = min(quality_ok_only, key=lambda r: r["total_energy_kJ"])
    single_rule = "no single model respects budget and capacity: lowest energy among the models eligible for every request"
else:
    best_single = min(singles.values(), key=lambda r: (r["not_eligible_requests"], r["total_energy_kJ"]))
    single_rule = "no model is eligible for every request: the model eligible for the most requests (quality violated)"
DECISION_SECONDS["Single model"] = time.perf_counter() - t0
results[best_single["method"]] = best_single
print(f"{best_single['method']}: {single_rule}")
show(pd.DataFrame(singles.values()).set_index("method")[["total_energy_kJ", "cost_usd", "not_eligible_requests", "quality_ok",
                                                         "budget_ok", "capacity_ok", "max_utilisation"]].round(4),
     "Every model alone")


Single model: llama-3.3-70b-instruct: lowest energy among the models that respect every constraint

=== Every model alone
                                                  total_energy_kJ  cost_usd  not_eligible_requests  quality_ok  budget_ok  capacity_ok  max_utilisation
method                                                                                                                                                 
Single model: llama3-openbiollm-70b                      163.1840    0.0450                     52       False      False         True           0.1170
Single model: llama3-med42-70b                           163.1852    0.0450                     52       False      False         True           0.1172
Single model: llama-3.1-70b-instruct                     163.1852    0.0548                      6       False      False         True           0.1172
Single model: llama-3.1-8b-instruct                       38.6244    0.0108                    162       False       T

### Greedy facility location

Starting from no active model, adds the model that most reduces the total energy until no addition reduces it; budget and capacity are checked afterwards.

In [46]:
def greedy_energy(active):
    if not active:
        return np.inf
    per_req = [min((E_net.loc[i, j] for j in A[i] if j in active), default=np.inf) for i in REQUEST_IDS]
    uncovered = sum(np.isinf(per_req))
    return sum(p for p in per_req if np.isfinite(p)) + F[list(active)].sum() + uncovered * 1e12

t0 = time.perf_counter()
active = set()
while True:
    current = greedy_energy(active)
    best_add = min((j for j in MODEL_IDS if j not in active), key=lambda j: greedy_energy(active | {j}), default=None)
    if best_add is None or greedy_energy(active | {best_add}) >= current:
        break
    active.add(best_add)
greedy = pd.Series({i: min((j for j in A[i] if j in active), key=lambda j: E_net.loc[i, j]) for i in REQUEST_IDS})
DECISION_SECONDS["Greedy facility location"] = time.perf_counter() - t0
results["Greedy facility location"] = evaluate(greedy, "Greedy facility location")
print("Greedy active models:", [short(j) for j in active])

Greedy active models: ['llama-3.3-70b-instruct']


## Comparison

### Lower bound of the relaxed problem

The best of the $2^K - 1$ subsets of models, without budget and capacity:

$$\mathrm{Gap} = \frac{E_{\mathrm{method}} - E_{\mathrm{LB}}}{E_{\mathrm{LB}}}$$

In [47]:
def relaxed_lower_bound(F_, E_net_, eligible_):
    models_ = list(eligible_.columns)
    Ev = np.where(eligible_.values, E_net_[models_].values, np.inf)
    Fv = np.asarray([F_[j] for j in models_], dtype=float)
    best, best_sub = np.inf, None
    for mask in range(1, 2 ** len(models_)):
        sub = np.array([(mask >> k) & 1 for k in range(len(models_))], dtype=bool)
        per_req = np.where(sub[None, :], Ev, np.inf).min(axis=1)
        if np.isinf(per_req).any():
            continue
        total = per_req.sum() + Fv[sub].sum()
        if total < best:
            best, best_sub = total, [models_[k] for k in range(len(models_)) if sub[k]]
    return best, best_sub

t0 = time.perf_counter()
E_LB, lb_subset = relaxed_lower_bound(F, E_net, eligible)
LOWER_BOUND_SECONDS = time.perf_counter() - t0
E_LB_kJ = E_LB / 1e3
print(f"Lower bound of the relaxed problem: {E_LB_kJ:.3f} kJ with {[short(j) for j in lb_subset]}")

Lower bound of the relaxed problem: 174.057 kJ with ['llama-3.3-70b-instruct']


### Feasible and infeasible solutions

In [48]:
comp = pd.DataFrame(results.values()).set_index("method")
num = ["total_energy_kJ", "activation_kJ", "serving_kJ", "parking_kJ", "cost_usd", "mean_quality", "models", "gpus",
       "to_specialists", "not_eligible_requests", "serving_s", "max_utilisation", "energy_reference_length_kJ", "cost_reference_length_usd"]
comp[num] = comp[num].astype(float)
for k in ["quality_ok", "budget_ok", "capacity_ok"]:
    comp[k] = comp[k].astype(float)       # share of feasible runs for Random, 0 or 1 for the other methods
comp["feasible"] = (comp["quality_ok"] >= 1 - 1e-9) & (comp["budget_ok"] >= 1 - 1e-9) & (comp["capacity_ok"] >= 1 - 1e-9)
comp["gap_to_LB_%"] = (comp["total_energy_kJ"] - E_LB_kJ) / E_LB_kJ * 100
E_milp = comp.loc["Facility-location MILP", "total_energy_kJ"]
comp["energy_vs_MILP_%"] = (comp["total_energy_kJ"] - E_milp) / E_milp * 100

def violated(r):
    out = []
    if r["quality_ok"] < 1 - 1e-9: out.append(f"quality ({r['not_eligible_requests']:.0f} requests to a model not eligible)")
    if r["budget_ok"] < 1 - 1e-9: out.append(f"budget (cost {r['cost_usd']:.4f} > B {B:.4f})")
    if r["capacity_ok"] < 1 - 1e-9: out.append(f"capacity (max share of T used {r['max_utilisation']:.2f})")
    return "; ".join(out)
comp["violates"] = comp.apply(violated, axis=1)
comp["decision_time_s"] = [DECISION_SECONDS.get("Single model" if m.startswith("Single model") else m, np.nan) for m in comp.index]

cols = ["total_energy_kJ", "activation_kJ", "serving_kJ", "parking_kJ", "cost_usd", "mean_quality", "models", "gpus",
        "to_specialists", "max_utilisation", "gap_to_LB_%", "energy_vs_MILP_%", "energy_reference_length_kJ",
        "feasible_reference_length", "decision_time_s", "active"]
feasible = comp[comp["feasible"]].sort_values("total_energy_kJ")
infeasible = comp[~comp["feasible"]].sort_values("total_energy_kJ")
show(feasible[cols].round(3), "A. Feasible solutions (quality, budget and capacity respected)")
show(infeasible[cols + ["violates"]].round(3), "B. Infeasible solutions (shown for reference only: their energy is not comparable)")
print(f"\n=== C. Theoretical reference\nLower bound of the relaxed problem: {E_LB_kJ:.3f} kJ with {[short(j) for j in lb_subset]}")
print(f"\n=== D. Time (s): data preparation {PREPARATION_SECONDS:.3f} | budget {BUDGET_SECONDS:.3f} | "
      f"MILP building {MILP_BUILD_SECONDS:.3f} + solving {MILP_SOLVE_SECONDS:.3f} | lower bound {LOWER_BOUND_SECONDS:.3f}")
print(f"MILP status: {MILP_STATUS}")



=== A. Feasible solutions (quality, budget and capacity respected)
                                      total_energy_kJ  activation_kJ  serving_kJ  parking_kJ  cost_usd  mean_quality  models  gpus  to_specialists  max_utilisation  gap_to_LB_%  energy_vs_MILP_%  energy_reference_length_kJ feasible_reference_length  decision_time_s                                                                                                        active
method                                                                                                                                                                                                                                                                                                                                                                                 
Facility-location MILP                        174.057          9.396      59.652     105.010     0.040         0.539     1.0   4.0             0.0            0.123        0.000    

## Checks

In [49]:
assert milp_assign.notna().all() and len(milp_assign) == N, "some request is not assigned"
assert all(eligible.loc[i, j] for i, j in milp_assign.items()), "an assignment is not eligible"
assert all(j in M_star for j in milp_assign.values), "a request goes to an inactive model"
if MILP_STATUS == "optimal":
    assert comp.loc["Facility-location MILP", "feasible"], "the MILP solution violates a constraint"
else:
    print(f"MILP {MILP_STATUS}: feasibility check skipped")
assert E_milp >= E_LB_kJ - 1e-6, "the MILP is below the lower bound"
if MILP_STATUS == "optimal":
    assert all(E_milp <= comp.loc[m, "total_energy_kJ"] + 1e-6 for m in feasible.index), "a feasible baseline beats the MILP"
print("All checks passed.")

All checks passed.


## Synthetic test (optimum computed by hand)

Two models and three requests, solved with the same functions as the FL-MILP for a loose budget (106 kJ), a tight budget (134 kJ), a tight capacity (134 kJ) and an infeasible budget, whose optima were computed by hand.

In [50]:
def synthetic_case(B_, time_G):
    mods, reqs = ["G", "P"], ["r1", "r2", "r3"]
    F_ = pd.Series({"G": 100.0, "P": 30.0})
    E_ = pd.DataFrame({"G": [2.0, 2.0, 2.0], "P": [1.0, 1.0, 1.0]}, index=reqs)
    c_ = pd.DataFrame({"G": [10.0, 10.0, 10.0], "P": [2.0, 2.0, 2.0]}, index=reqs)
    t_ = pd.DataFrame({"G": [1.0, 1.0, 1.0], "P": [1.0, 1.0, 1.0]}, index=reqs)
    elig = pd.DataFrame({"G": [True, True, True], "P": [True, True, False]}, index=reqs)
    A_ = {i: [j for j in mods if elig.loc[i, j]] for i in reqs}
    p_ = create_problem("synthetic")
    y_, x_, pairs_ = add_variables(mods, reqs, A_)
    add_objective(p_, y_, x_, pairs_, F_, E_)
    add_assignment(p_, x_, reqs, A_)
    add_linking(p_, x_, y_, pairs_)
    add_budget(p_, x_, y_, pairs_, c_, pd.Series(0.0, index=mods), B_)
    add_capacity(p_, x_, y_, pairs_, t_, pd.Series({"G": time_G, "P": 10.0}))
    status_, culprits_ = solve(p_, len(mods))
    lb_, _ = relaxed_lower_bound(F_, E_, elig)
    if status_ != "Optimal":
        return status_, culprits_, None, None, None, lb_
    assign_ = {i: j for (i, j), v in x_.items() if v.value() > 0.5}
    active_ = sorted(j for j in mods if y_[j].value() > 0.5)
    energy_ = sum(F_[j] for j in active_) + sum(E_.loc[i, j] for i, j in assign_.items())      # recomputed by hand
    cost_ = sum(c_.loc[i, j] for i, j in assign_.items())
    assert abs(energy_ - pulp.value(p_.objective)) < 1e-9, "objective differs from the energy recomputed by hand"
    return status_, culprits_, active_, assign_, (energy_, cost_), lb_

expected = {
    "A. loose budget":   dict(B=100, time_G=10.0, active=["G"], to_P=[], energy=106, cost=30),
    "B. tight budget":   dict(B=20, time_G=10.0, active=["G", "P"], to_P=["r1", "r2"], energy=134, cost=14),
    "C. tight capacity": dict(B=100, time_G=2.5, active=["G", "P"], to_P=["r1", "r2"], energy=134, cost=14),
    "D. infeasible":     dict(B=10, time_G=10.0, culprit="budget"),
}
rows = []
for name, e in expected.items():
    status_, culprits_, active_, assign_, ec, lb_ = synthetic_case(e["B"], e["time_G"])
    assert abs(lb_ - 106) < 1e-9, f"{name}: lower bound {lb_} instead of 106"
    if "culprit" in e:
        assert status_ != "Optimal" and culprits_ == [e["culprit"]], f"{name}: expected infeasible because of {e['culprit']}, got {status_} {culprits_}"
        rows.append({"case": name, "status": status_, "active": "-", "to_P": "-", "energy_kJ": "-", "cost": "-", "diagnosis": culprits_})
        continue
    to_P = sorted(i for i, j in assign_.items() if j == "P")
    assert status_ == "Optimal" and active_ == e["active"] and to_P == e["to_P"], f"{name}: wrong solution {active_} {assign_}"
    assert abs(ec[0] - e["energy"]) < 1e-9 and abs(ec[1] - e["cost"]) < 1e-9, f"{name}: energy or cost differ from the hand computation"
    rows.append({"case": name, "status": status_, "active": active_, "to_P": to_P, "energy_kJ": ec[0], "cost": ec[1], "diagnosis": "-"})
show(pd.DataFrame(rows).set_index("case"), "Synthetic test: solutions found by the MILP functions")
print("Synthetic test passed: the MILP functions reproduce the optimum computed by hand in all four cases.")


=== Synthetic test: solutions found by the MILP functions
                       status  active      to_P energy_kJ  cost diagnosis
case                                                                     
A. loose budget       Optimal     [G]        []     106.0  30.0         -
B. tight budget       Optimal  [G, P]  [r1, r2]     134.0  14.0         -
C. tight capacity     Optimal  [G, P]  [r1, r2]     134.0  14.0         -
D. infeasible      Infeasible       -         -         -     -  [budget]
Synthetic test passed: the MILP functions reproduce the optimum computed by hand in all four cases.


## Save

In [51]:
run_id = f"N{N}_seed{SEED}_T{WORKLOAD_DURATION_S}_batch{BATCH}_{QUALITY_SCALE}" + ("_verbosity" if USE_VERBOSITY else "")
comp.to_csv(os.path.join(OUTPUT_DIR, f"comparison_{run_id}.csv"))
assignment.to_csv(os.path.join(OUTPUT_DIR, f"assignment_{run_id}.csv"))
selection.to_csv(os.path.join(OUTPUT_DIR, f"selection_{run_id}.csv"))
milp_timeline.to_csv(os.path.join(OUTPUT_DIR, f"timeline_{run_id}.csv"))
milp_timeline_ref.to_csv(os.path.join(OUTPUT_DIR, f"timeline_reference_length_{run_id}.csv"))
spread.to_csv(os.path.join(OUTPUT_DIR, f"random_spread_{run_id}.csv"))
config = {k: val for k, val in globals().items()
          if k.isupper() and len(k) > 2 and k not in ("REQUEST_IDS", "MODEL_IDS")
          and isinstance(val, (int, float, str, list, dict, type(None)))}
with open(os.path.join(OUTPUT_DIR, f"config_{run_id}.json"), "w") as f:
    json.dump(config, f, indent=2, default=str)
print("Saved run", run_id, "in", OUTPUT_DIR)

Saved run N200_seed42_T600_batch128_absolute_verbosity in /content/drive/MyDrive/thesis/pipeline_v9
